# 02. 임베딩 테스트
임베딩 품질과 벡터 검색 성능을 검증합니다.

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from src.embedding.embedder import MedicalEmbedder
from src.retrieval.vector_store import get_chroma_client, get_or_create_collection
from src.retrieval.retriever import retrieve
from dotenv import load_dotenv
load_dotenv("../.env")

## 임베딩 모델 로드

In [ ]:
embedder = MedicalEmbedder()
print("임베딩 모델 로드 완료")

## 유사도 테스트

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

pairs = [
    ("diabetes treatment", "type 2 diabetes medication"),   # 유사해야 함
    ("diabetes treatment", "cancer surgery"),               # 달라야 함
    ("insulin therapy", "blood glucose control"),           # 유사해야 함
]

for a, b in pairs:
    va = embedder.embed_single(a)
    vb = embedder.embed_single(b)
    sim = cosine_similarity([va], [vb])[0][0]
    print(f"{sim:.3f} | {a} ↔ {b}")

## 벡터 DB 검색 테스트

In [ ]:
client = get_chroma_client()
collection = get_or_create_collection(client)
print(f"총 청크 수: {collection.count()}")

test_queries = [
    "What are the treatment options for type 2 diabetes?",
    "side effects of metformin",
    "insulin resistance mechanism",
]

for query in test_queries:
    q_emb = embedder.embed_single(query)
    results = retrieve(collection, q_emb, top_k=3)
    print(f"
쿼리: {query}")
    for r in results:
        print(f"  유사도: {r['score']:.4f} | {r['metadata']['title'][:60]}...")

## PCA 시각화

In [ ]:
sample_texts = [
    "diabetes treatment metformin",
    "insulin blood glucose",
    "cancer chemotherapy",
    "heart disease cardiovascular",
    "diabetes complications",
    "hypertension blood pressure",
]
labels = ["DM치료","인슐린","암","심장","DM합병증","고혈압"]

vecs = np.array([embedder.embed_single(t) for t in sample_texts])
pca = PCA(n_components=2)
reduced = pca.fit_transform(vecs)

plt.figure(figsize=(8,6))
plt.scatter(reduced[:,0], reduced[:,1])
for i, label in enumerate(labels):
    plt.annotate(label, (reduced[i,0], reduced[i,1]), fontsize=12)
plt.title("임베딩 PCA 시각화")
plt.tight_layout()
plt.show()